---
title: "Predictive and Forecasting Models for Olist" 
author: "Hoang Son Lai" 
date: "06/23/2026" 
format: 
 html: 
  toc: true 
  css: styles.css 
  embed-resources: true 
  code-fold: true
 ---

In [7]:
# Cell 1 - Setup
import os, warnings, json
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

DB = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "ecom.duckdb"))
con = duckdb.connect(DB, read_only=True)

plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "axes.labelcolor":  "#e6edf3",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "text.color":       "#e6edf3",
    "grid.color":       "#30363d",
    "grid.linewidth":   0.5,
})
PALETTE = ["#f0883e","#58a6ff","#3fb950","#f85149","#d29922","#8b949e"]
print("Setup OK")

Setup OK


In [8]:
# Cell 2 - Table row counts
tables = [
    "bronze_orders", "bronze_customers", "bronze_order_items",
    "bronze_reviews", "bronze_products", "bronze_sellers",
    "bronze_payments", "bronze_geolocation",
]
for t in tables:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<35} {n:>8,} rows")

  bronze_orders                         99,441 rows
  bronze_customers                      99,441 rows
  bronze_order_items                   112,650 rows
  bronze_reviews                        99,224 rows
  bronze_products                       32,951 rows
  bronze_sellers                         3,095 rows
  bronze_payments                      103,886 rows
  bronze_geolocation                    19,015 rows


In [9]:
# Cell 3 - Order status and date range
df_status = con.execute("""
    SELECT order_status,
           COUNT(*) AS n,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct,
           MIN(purchased_at) AS first_order,
           MAX(purchased_at) AS last_order
    FROM main_silver.stg_orders
    GROUP BY order_status
    ORDER BY n DESC
""").df()
print(df_status.to_string(index=False))

order_status     n   pct         first_order          last_order
   delivered 96478 98.24 2016-09-15 12:16:38 2018-08-29 15:00:37
     shipped  1107  1.13 2016-09-04 21:15:19 2018-09-03 09:06:57
    invoiced   314  0.32 2016-10-04 13:02:10 2018-08-14 18:45:08
  processing   301  0.31 2016-10-05 22:44:13 2018-07-23 18:03:03
     created     5  0.01 2017-11-06 13:12:34 2018-02-09 17:21:04
    approved     2  0.00 2017-02-06 20:18:17 2017-04-25 01:25:34
